# Watermarked text with GPT-2: generation, detection and Chapter 2 figures

This notebook produces the project's main dataset.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. When asked, upload **`human_texts.csv`** (from the project folder).
3. At the end, a zip file with the results and figures downloads automatically.

It runs in about 10–20 minutes on a free Colab GPU for all 498 prompts.
Settings are in the *Settings* cell; keep them unchanged unless the team agrees, so all numbers match.

## 1. Project code
These cells write the shared code files. They are identical to the `.py` files in the project folder.

In [ ]:
%%writefile watermark_core.py
"""
watermark_core.py - the shared heart of the project.

Both the WATERMARK (generation) and the DETECTOR must use exactly these
functions, otherwise the detector rebuilds different green lists.

Method: the "soft red list" watermark of Kirchenbauer et al. (ICML 2023),
using the previous token as context.

  previous token + secret key --> seed --> shuffle vocabulary
  --> first gamma * |V| tokens of the shuffle are GREEN, the rest RED

Only numpy and scipy are needed here (no PyTorch), so the detector can run
anywhere.
"""

from functools import lru_cache
import hashlib

import numpy as np
from scipy import stats

# The team's secret key. Anyone holding it can detect the watermark.
SECRET_KEY = 15485863
GAMMA = 0.5


def _seed(prev_token_id: int, key: int) -> int:
    """Combine the secret key and the previous token into one 64-bit seed.

    SHA-256 is a cryptographic hash: changing the key or the token by 1
    gives a completely unrelated seed (this is the real-world version of the
    'mod 101' formula in our toy example).
    """
    digest = hashlib.sha256(f"{key}:{int(prev_token_id)}".encode()).digest()
    return int.from_bytes(digest[:8], "big")


@lru_cache(maxsize=2048)
def _green_mask_cached(prev_token_id: int, vocab_size: int, gamma: float, key: int) -> np.ndarray:
    rng = np.random.default_rng(_seed(prev_token_id, key))
    order = rng.permutation(vocab_size)          # the random shuffle
    n_green = int(gamma * vocab_size)            # e.g. 0.5 * 50257 = 25128
    mask = np.zeros(vocab_size, dtype=bool)
    mask[order[:n_green]] = True                 # first part of the shuffle = green
    mask.setflags(write=False)                   # cached arrays must never be modified
    return mask


def green_mask(prev_token_id: int, vocab_size: int, gamma: float = GAMMA,
               key: int = SECRET_KEY) -> np.ndarray:
    """Boolean array of length vocab_size: True where the token is GREEN
    at a position whose previous token is prev_token_id."""
    return _green_mask_cached(int(prev_token_id), int(vocab_size), float(gamma), int(key))


def is_green(prev_token_id: int, token_id: int, vocab_size: int,
             gamma: float = GAMMA, key: int = SECRET_KEY) -> bool:
    return bool(green_mask(prev_token_id, vocab_size, gamma, key)[int(token_id)])


def green_flags(token_ids, vocab_size: int, context_token_id=None,
                gamma: float = GAMMA, key: int = SECRET_KEY,
                ignore_repeated_bigrams: bool = False):
    """Colour every token of a text.

    token_ids        : the text's token ids
    context_token_id : the token just before the text (e.g. last prompt token).
                       If None, the first token is skipped (it has no previous token).
    ignore_repeated_bigrams : if True, a (previous token, token) pair that already
                       appeared is not scored again. Repeated pairs always get the
                       same colour, so they are not independent coin flips.

    Returns a list with one entry per token: True (green), False (red) or
    None (not scored).
    """
    ids = [int(t) for t in token_ids]
    flags = []
    seen = set()
    prev = None if context_token_id is None else int(context_token_id)
    for tok in ids:
        if prev is None:
            flags.append(None)
        elif ignore_repeated_bigrams and (prev, tok) in seen:
            flags.append(None)
        else:
            seen.add((prev, tok))
            flags.append(is_green(prev, tok, vocab_size, gamma, key))
        prev = tok
    return flags


def detect(token_ids, vocab_size: int, context_token_id=None,
           gamma: float = GAMMA, key: int = SECRET_KEY,
           ignore_repeated_bigrams: bool = False) -> dict:
    """The hypothesis test.

    H0: the text was written without knowledge of the key, so each scored
        token is green with probability gamma and G ~ Binomial(T, gamma).
    H1: the text is watermarked (green more often than gamma). One-sided.
    """
    flags = green_flags(token_ids, vocab_size, context_token_id, gamma, key,
                        ignore_repeated_bigrams)
    scored = [f for f in flags if f is not None]
    T = len(scored)
    G = int(sum(scored))
    if T == 0:
        return dict(T=0, G=0, green_frac=np.nan, z=np.nan, p_z=np.nan, p_exact=np.nan)
    z = (G - gamma * T) / np.sqrt(T * gamma * (1 - gamma))
    return dict(
        T=T,
        G=G,
        green_frac=G / T,
        z=float(z),
        p_z=float(stats.norm.sf(z)),                    # normal approximation
        p_exact=float(stats.binom.sf(G - 1, T, gamma)), # exact P(G' >= G) under H0
    )

In [ ]:
%%writefile gpt2_watermark.py
"""
gpt2_watermark.py - watermarked text generation with GPT-2 (Hugging Face).

Needs: torch, transformers, numpy, scipy  (all preinstalled on Google Colab).
Uses watermark_core.py for the green lists, so generation and detection
always agree.
"""

import numpy as np
import torch
from transformers import LogitsProcessor, LogitsProcessorList

from watermark_core import GAMMA, SECRET_KEY, detect, green_mask


class WatermarkLogitsProcessor(LogitsProcessor):
    """Adds delta to the score (logit) of every GREEN token before sampling.

    Hugging Face calls this once per generation step with
      input_ids : (batch, tokens so far)
      scores    : (batch, vocab_size) raw scores for the next token
    """

    def __init__(self, vocab_size: int, gamma: float = GAMMA, delta: float = 2.0,
                 key: int = SECRET_KEY):
        self.vocab_size = vocab_size
        self.gamma = gamma
        self.delta = delta
        self.key = key

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if self.delta == 0:
            return scores
        if scores.shape[-1] != self.vocab_size:
            raise ValueError(f"scores have {scores.shape[-1]} columns, expected {self.vocab_size}")
        for b in range(input_ids.shape[0]):
            prev = int(input_ids[b, -1])
            mask = torch.from_numpy(np.array(green_mask(prev, self.vocab_size, self.gamma, self.key)))
            scores[b, mask.to(scores.device)] += self.delta
        return scores


def generate_batch(model, tokenizer, prompts, delta, n_tokens=200, gamma=GAMMA,
                   key=SECRET_KEY, seed=0, device=None):
    """Continue each prompt by exactly n_tokens tokens.

    Plain multinomial sampling (temperature 1, no top-k / top-p), as in the
    watermark paper. delta = 0 gives ordinary, unwatermarked AI text.
    Returns a list of dicts with the generated token ids and text.
    """
    device = device or next(model.parameters()).device
    vocab_size = model.config.vocab_size
    tokenizer.padding_side = "left"              # GPT-2 must be left-padded for batching
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    enc = tokenizer(list(prompts), return_tensors="pt", padding=True).to(device)
    torch.manual_seed(seed)
    with torch.no_grad():
        out = model.generate(
            **enc,
            do_sample=True,
            top_k=0,
            top_p=1.0,
            temperature=1.0,
            max_new_tokens=n_tokens,
            min_new_tokens=n_tokens,             # forbid early end-of-text: fixed length
            pad_token_id=tokenizer.pad_token_id,
            logits_processor=LogitsProcessorList(
                [WatermarkLogitsProcessor(vocab_size, gamma, delta, key)]),
        )
    new_tokens = out[:, enc["input_ids"].shape[1]:].cpu().tolist()
    return [dict(gen_ids=ids, gen_text=tokenizer.decode(ids)) for ids in new_tokens]


def detect_text(text, prompt, tokenizer, vocab_size, n_tokens=200, gamma=GAMMA,
                key=SECRET_KEY, ignore_repeated_bigrams=False):
    """What a real detector does: it only sees TEXT.

    Re-tokenizes the text, keeps its first n_tokens tokens and uses the last
    prompt token as the context for the first one.
    """
    prompt_ids = tokenizer(prompt)["input_ids"]
    ids = tokenizer(text)["input_ids"][:n_tokens]
    res = detect(ids, vocab_size, context_token_id=prompt_ids[-1], gamma=gamma,
                 key=key, ignore_repeated_bigrams=ignore_repeated_bigrams)
    res["token_ids"] = ids
    res["context_id"] = prompt_ids[-1]
    return res


def next_token_distribution(model, tokenizer, context, delta=2.0, gamma=GAMMA,
                            key=SECRET_KEY, top=10):
    """Probabilities of the top next tokens before and after the watermark boost
    (for the Chapter 2 bar chart)."""
    device = next(model.parameters()).device
    ids = tokenizer(context, return_tensors="pt")["input_ids"].to(device)
    with torch.no_grad():
        logits = model(ids).logits[0, -1].float().cpu().numpy()
    mask = np.array(green_mask(int(ids[0, -1]), model.config.vocab_size, gamma, key))
    boosted = logits + delta * mask
    p_before = _softmax(logits)
    p_after = _softmax(boosted)
    order = np.argsort(-p_before)[:top]
    rows = [dict(token=tokenizer.decode([int(i)]), green=bool(mask[i]),
                 p_before=float(p_before[i]), p_after=float(p_after[i])) for i in order]
    return rows, float(p_before[mask].sum()), float(p_after[mask].sum())


def _softmax(x):
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

In [ ]:
%%writefile figures.py
"""
figures.py - presentation figures for Chapter 2.

  coloured_text_figure : text with every token shaded green / red
                         (human text vs watermarked text, side by side)
  probability_bars     : top next-token probabilities before vs after the boost

Works with any tokenizer: pass the token STRINGS (with their leading spaces)
and the green/red flags from watermark_core.green_flags.
Green tokens are also drawn in bold, so the figure still works for
colour-blind viewers and in black-and-white print.
"""

import textwrap

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Patch

GREEN_FILL = "#9fdcb0"
RED_FILL = "#f6c1b8"
NONE_FILL = "#e6e6e6"
INK = "#1f2933"
MUTED = "#6b7280"
BEFORE = "#c4c9d0"
AFTER = "#33658a"


def _draw_tokens(ax, tokens, flags, fontsize=11, y_top=0.86, max_width=0.98):
    """Lay out tokens like a paragraph. The figure layout must be FINAL before
    calling this, because positions are measured in axes coordinates."""
    fig = ax.figure
    renderer = fig.canvas.get_renderer()
    inv = ax.transAxes.inverted()

    def extent(artist):
        (x0, y0), (x1, y1) = inv.transform(artist.get_window_extent(renderer))
        return x0, y0, x1, y1

    probe = ax.text(0, 0, "Mg", fontsize=fontsize, transform=ax.transAxes)
    _, y0, _, y1 = extent(probe)
    probe.remove()
    line_h = 1.75 * (y1 - y0)
    a = ax.text(0, 0, "a a", fontsize=fontsize, transform=ax.transAxes)
    b = ax.text(0, 0, "aa", fontsize=fontsize, transform=ax.transAxes)
    space_w = (extent(a)[2] - extent(a)[0]) - (extent(b)[2] - extent(b)[0])
    a.remove(); b.remove()
    x, y = 0.0, y_top
    for tok, flag in zip(tokens, flags):
        lead_space = tok[:1] in (" ", "\n")
        word = tok.strip() or "\u00b7"
        if lead_space and x > 0:
            x += space_w
        fill = NONE_FILL if flag is None else (GREEN_FILL if flag else RED_FILL)
        txt = ax.text(x, y, word, fontsize=fontsize, color=INK, weight="bold" if flag else "normal",
                      transform=ax.transAxes, va="baseline", ha="left", zorder=3)
        x0, b0, x1, b1 = extent(txt)
        if x1 > max_width and x > 0:            # wrap to the next line
            x, y = 0.0, y - line_h
            txt.set_position((x, y))
            x0, b0, x1, b1 = extent(txt)
        ax.add_patch(FancyBboxPatch((x0 - 0.001, b0 - 0.006), (x1 - x0) + 0.002, (b1 - b0) + 0.012,
                                    boxstyle="round,pad=0,rounding_size=0.003",
                                    facecolor=fill, edgecolor="none",
                                    transform=ax.transAxes, zorder=2))
        x = x1 + 0.0015
    return y, line_h


def coloured_text_figure(panels, path, n_show=70, fontsize=11):
    """panels: list of dicts with keys title, tokens (strings), flags, stats (dict from detect).
    Shows the first n_show tokens of each text; the stats line covers the whole text."""
    fig, axes = plt.subplots(1, len(panels), figsize=(7.2 * len(panels), 3.5))
    if len(panels) == 1:
        axes = [axes]
    fig.subplots_adjust(left=0.02, right=0.98, top=0.97, bottom=0.03, wspace=0.08)
    fig.canvas.draw()                              # fix the layout BEFORE measuring text
    for ax, p in zip(axes, panels):
        ax.set_axis_off()
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.text(0.0, 0.99, p["title"], fontsize=fontsize + 3, weight="bold", color=INK,
                transform=ax.transAxes, va="top")
        y, line_h = _draw_tokens(ax, p["tokens"][:n_show], p["flags"][:n_show], fontsize=fontsize, y_top=0.80)
        shown = [f for f in p["flags"][:n_show] if f is not None]
        s = p["stats"]
        ax.text(0.0, y - 1.6 * line_h,
                f"Shown: first {len(p['tokens'][:n_show])} tokens ({sum(shown)} green).   "
                f"Whole text: {s['G']} of {s['T']} green ({100 * s['green_frac']:.0f}%), "
                f"z = {s['z']:.2f}, exact p = {s['p_exact']:.2g}",
                fontsize=fontsize - 1, color=MUTED, transform=ax.transAxes)
    handles = [Patch(facecolor=GREEN_FILL, label="green token (bold)"),
               Patch(facecolor=RED_FILL, label="red token")]
    axes[-1].legend(handles=handles, loc="upper right", ncol=2, frameon=False,
                    bbox_to_anchor=(1.0, 1.03), fontsize=fontsize - 1)
    fig.savefig(path, dpi=200, bbox_inches="tight", pad_inches=0.15)
    plt.close(fig)


def probability_bars(rows, green_before, green_after, context, delta, path):
    """rows: list of dicts token, green, p_before, p_after (top tokens by p_before)."""
    labels = [f"{r['token'].strip() or repr(r['token'])}\n({'G' if r['green'] else 'R'})" for r in rows]
    xs = range(len(rows))
    w = 0.38
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar([x - w / 2 - 0.01 for x in xs], [100 * r["p_before"] for r in rows], w,
           color=BEFORE, label="before watermark")
    ax.bar([x + w / 2 + 0.01 for x in xs], [100 * r["p_after"] for r in rows], w,
           color=AFTER, label=f"after adding δ = {delta:g} to green tokens")
    ax.set_xticks(list(xs))
    ax.set_xticklabels(labels, fontsize=11)
    for tick, r in zip(ax.get_xticklabels(), rows):
        tick.set_bbox(dict(facecolor=GREEN_FILL if r["green"] else RED_FILL,
                           edgecolor="none", boxstyle="round,pad=0.25"))
    ax.set_ylabel("Probability of being the next token (%)", color=INK)
    ctx = textwrap.shorten(context, 70, placeholder="…")
    ax.set_title(f"Next token after “…{ctx[-60:]}”\n"
                 f"Total chance of a green token: {100 * green_before:.0f}% → {100 * green_after:.0f}%",
                 fontsize=13, color=INK, loc="left")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_color(MUTED); ax.spines["bottom"].set_color(MUTED)
    ax.tick_params(colors=MUTED)
    ax.yaxis.grid(True, color="#eeeeee"); ax.set_axisbelow(True)
    ax.legend(frameon=False, fontsize=11)
    fig.text(0.01, 0.005, "(G) = green token, (R) = red token for this step", fontsize=9, color=MUTED)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)

In [ ]:
%%writefile sanity_checks.py
"""
sanity_checks.py - checks a results file before it is handed to the analysts.

Works on both results files:
  python sanity_checks.py prototype/results_prototype.csv
  python sanity_checks.py data/results_gpt2.csv

What it checks (and what we expect):
  1. every text has the intended length T
  2. unwatermarked texts (human, AI with delta = 0) are about 50% green,
     z has mean ~0 and SD ~1, and about 5% are flagged at alpha = 0.05
  3. watermarked texts are clearly greener, more so for delta = 2 than delta = 1
  4. the same with repeated (previous token, token) pairs counted once
Writes a short report next to the input file.
"""

import os
import sys

import numpy as np
import pandas as pd
from scipy import stats


def summarise(d):
    d = d.copy()
    d["setting"] = np.where(d.source == "human", "human",
                            "AI, delta = " + d.delta.fillna(-1).astype(int).astype(str))
    rows = []
    for name, g in d.groupby("setting", sort=False):
        fp = (g.z > 1.645).mean()
        n = len(g)
        # one-proportion test: is the flag rate at alpha = 0.05 really 5%?
        # (only meaningful for unwatermarked text; for watermarked text the flag rate is the power)
        watermarked = name.startswith("AI") and not name.endswith("= 0")
        p_cal = np.nan if watermarked else stats.binomtest(int((g.z > 1.645).sum()), n, 0.05).pvalue
        rows.append({
            "setting": name, "texts": n, "mean T": round(g["T"].mean(), 1),
            "mean green %": round(100 * g.green_frac.mean(), 1),
            "mean z": round(g.z.mean(), 2), "SD of z": round(g.z.std(ddof=1), 2),
            "flagged at z > 1.645": f"{100 * fp:.1f}%",
            "p (flag rate = 5%)": "" if np.isnan(p_cal) else f"{p_cal:.2g}",
            "flagged at z > 4": f"{100 * (g.z > 4).mean():.1f}%",
            "mean z (dedup)": round(g.z_dedup.mean(), 2),
            "flagged at z > 1.645 (dedup)": f"{100 * (g.z_dedup > 1.645).mean():.1f}%",
        })
    return pd.DataFrame(rows)


def _table(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:                      # 'tabulate' not installed
        return df.to_string(index=False)


def main(path):
    d = pd.read_csv(path)
    s = summarise(d)
    lines = [f"# Sanity checks for {os.path.basename(path)}", "",
             f"{len(d)} texts. Lengths: min T = {d['T'].min()}, max T = {d['T'].max()}.", "",
             _table(s), "",
             "Expected: human and delta = 0 rows near 50% green, mean z near 0, SD near 1, "
             "about 5% flagged at z > 1.645 and none at z > 4; delta = 1 and 2 rows far greener. "
             "'dedup' columns count each repeated (previous token, token) pair once."]
    report = "\n".join(lines)
    out = os.path.splitext(path)[0] + "_sanity_report.md"
    with open(out, "w") as fh:
        fh.write(report + "\n")
    print(report)
    print(f"\nreport written to {out}")


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else "prototype/results_prototype.csv")

## 2. Upload the human texts

In [ ]:
# [COLAB-ONLY]
import os
if not os.path.exists("human_texts.csv"):
    from google.colab import files
    files.upload()          # choose human_texts.csv

## 3. Settings

In [ ]:
N_PROMPTS = 498        # all Brown texts; use e.g. 60 for a quick test run
N_TOKENS = 200         # length of every text, in GPT-2 tokens
DELTAS = [0, 1, 2]     # 0 = ordinary AI text (control), 1 = weak, 2 = standard watermark
BATCH = 16
from watermark_core import GAMMA, SECRET_KEY
print("gamma =", GAMMA, "| key =", SECRET_KEY)

## 4. Load GPT-2

In [ ]:
# [MODEL-LOAD]
import torch
from transformers import AutoTokenizer, GPT2LMHeadModel
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").to(device).eval()
V = model.config.vocab_size
print("vocabulary size:", V)

## 5. Generate the AI texts
Each prompt is continued three times: δ = 0, 1 and 2. Fixed seeds make the run reproducible.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
from gpt2_watermark import generate_batch

human = pd.read_csv("human_texts.csv").head(N_PROMPTS)
generated = []
for delta in DELTAS:
    for start in tqdm(range(0, len(human), BATCH), desc=f"delta = {delta}"):
        chunk = human.iloc[start:start + BATCH]
        outs = generate_batch(model, tokenizer, list(chunk.prompt), delta=delta,
                              n_tokens=N_TOKENS, seed=10_000 * delta + start)
        for (_, r), o in zip(chunk.iterrows(), outs):
            generated.append(dict(text_id=r.text_id, delta=delta, text=o["gen_text"]))
print(len(generated), "AI texts generated")

## 6. Detect: score every text (human and AI) from its text only

In [ ]:
import csv
import os
from gpt2_watermark import detect_text

def score(text_id, genre, group, source, delta, prompt, text):
    r = detect_text(text, prompt, tokenizer, V, n_tokens=N_TOKENS)
    rd = detect_text(text, prompt, tokenizer, V, n_tokens=N_TOKENS, ignore_repeated_bigrams=True)
    return dict(text_id=text_id, genre=genre, group=group, source=source, delta=delta,
                T=r["T"], G=r["G"], green_frac=round(r["green_frac"], 4), z=round(r["z"], 4),
                p_z=r["p_z"], p_exact=r["p_exact"],
                T_dedup=rd["T"], G_dedup=rd["G"], z_dedup=round(rd["z"], 4),
                context_id=r["context_id"], token_ids=" ".join(map(str, r["token_ids"])), text=text)

info = human.set_index("text_id")
rows = []
for r in tqdm(human.itertuples(), total=len(human), desc="human"):
    rows.append(score(r.text_id, r.genre, r.group, "human", float("nan"), r.prompt, " " + r.human_text))
for g in tqdm(generated, desc="AI"):
    h = info.loc[g["text_id"]]
    rows.append(score(g["text_id"], h.genre, h.group, "ai", g["delta"], h.prompt, g["text"]))
results = pd.DataFrame(rows)
os.makedirs("outputs", exist_ok=True)
results.to_csv("outputs/results_gpt2.csv", index=False, quoting=csv.QUOTE_NONNUMERIC)
print(results.groupby(["source", "delta"], dropna=False)[["green_frac", "z"]].mean().round(3))

## 7. Sanity checks
Expected: human and δ = 0 texts about 50% green with mean z near 0; δ = 1 and δ = 2 clearly greener.

In [ ]:
import sanity_checks
sanity_checks.main("outputs/results_gpt2.csv")

## 8. Chapter 2 figures

In [ ]:
from IPython.display import Image, display
from figures import coloured_text_figure, probability_bars
from gpt2_watermark import next_token_distribution
from watermark_core import green_flags, detect

EXAMPLE_ID = "ca05" if "ca05" in set(results.text_id) else results.text_id.iloc[0]

def panel(source, delta, title):
    sel = results[(results.text_id == EXAMPLE_ID) & (results.source == source)]
    if delta is not None:
        sel = sel[sel.delta == delta]
    r = sel.iloc[0]
    ids = [int(t) for t in r.token_ids.split()]
    return dict(title=title,
                tokens=[tokenizer.decode([i]) for i in ids],
                flags=green_flags(ids, V, context_token_id=int(r.context_id)),
                stats=detect(ids, V, context_token_id=int(r.context_id)))

coloured_text_figure([panel("human", None, "Human (Brown corpus, 1961)"),
                      panel("ai", 2, "GPT-2 with watermark (\u03b4 = 2)")],
                     "outputs/fig_coloured_text_gpt2.png")
coloured_text_figure([panel("ai", 0, "GPT-2, no watermark (\u03b4 = 0)"),
                      panel("ai", 2, "GPT-2 with watermark (\u03b4 = 2)")],
                     "outputs/fig_coloured_text_gpt2_ai_vs_ai.png")

CONTEXT = "The jury said it did not find any evidence of"
rows_, g_before, g_after = next_token_distribution(model, tokenizer, CONTEXT, delta=2)
probability_bars(rows_, g_before, g_after, CONTEXT, 2, "outputs/fig_probability_bars_gpt2.png")
for f in ["fig_coloured_text_gpt2.png", "fig_coloured_text_gpt2_ai_vs_ai.png", "fig_probability_bars_gpt2.png"]:
    display(Image(f"outputs/{f}"))

## 9. Download everything

In [ ]:
# [COLAB-ONLY]
import shutil
from google.colab import files
shutil.make_archive("watermark_gpt2_outputs", "zip", "outputs")
files.download("watermark_gpt2_outputs.zip")